# 🏥 Apollo Hospital Voice AI Assistant

**Complete Standalone Notebook** - Real-time Voice AI for Indian Languages

Supports: Tamil, Telugu, Kannada, Hindi, English

## Pipeline Stages:
1. **VAD** - Voice Activity Detection (Silero)
2. **STT** - Speech-to-Text (Faster-Whisper large-v3)
3. **Language Detection** - Unicode script analysis
4. **Signal Extraction** - Intent, urgency, tone
5. **Safety Gate** - 5 escalation rules
6. **Policy Engine** - Hospital constraints
7. **LLM** - Response generation (LLaMA 3.1-8B 4-bit)
8. **TTS** - Text-to-Speech (Indic Parler-TTS)

## 📦 Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchaudio
!pip install -q faster-whisper
!pip install -q transformers accelerate bitsandbytes
!pip install -q parler-tts
!pip install -q soundfile librosa
!pip install -q ipywidgets

print("✅ All packages installed!")

## 🔧 Step 2: Setup & Device Configuration

In [ ]:
import os
import io
import time
import tempfile
import numpy as np
import torch
import soundfile as sf
from IPython.display import display, Audio, HTML, clear_output
import ipywidgets as widgets

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {DEVICE}")

if DEVICE == "cuda":
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Running on CPU - LLM inference will be slower")

# Set Hugging Face token (required for LLaMA)
# Replace with your token or set as environment variable
HF_TOKEN = os.environ.get("HUGGINGFACE_TOKEN", os.environ.get("HF_TOKEN", ""))
if not HF_TOKEN:
    print("⚠️  Set your Hugging Face token: HF_TOKEN environment variable")
    print("   Get token from: https://huggingface.co/settings/tokens")

## 📚 Step 3: Configuration & Keywords

In [ ]:
# ============================================================================
# LANGUAGE & HOSPITAL CONFIGURATION
# ============================================================================

SUPPORTED_LANGUAGES = {
    'kn': 'Kannada',
    'ta': 'Tamil', 
    'te': 'Telugu',
    'hi': 'Hindi',
    'en': 'English'
}

# Unicode script ranges for language detection
SCRIPT_RANGES = {
    'kn': [(0x0C80, 0x0CFF)],  # Kannada
    'ta': [(0x0B80, 0x0BFF)],  # Tamil
    'te': [(0x0C00, 0x0C7F)],  # Telugu
    'hi': [(0x0900, 0x097F)],  # Hindi/Devanagari
    'en': [(0x0041, 0x005A), (0x0061, 0x007A)],  # English A-Z, a-z
}

# Hospital Configuration
HOSPITAL_CONFIG = {
    "hospital_name": "Apollo Hospital",
    "city": "Bengaluru",
    "emergency_number": "108",
    "helpline": "1860-500-1066",
    "tone": "formal",
    "max_words": 50,
    "disclaimer_required": True,
    "disclaimer_text": "This is general information only. Please consult a doctor for proper medical advice.",
    "always_recommend_doctor": True
}

# Multi-language keyword dictionaries for Layer-1 signals
INTENT_KEYWORDS = {
    'medical_query': {
        'en': ['pain', 'fever', 'headache', 'cough', 'cold', 'stomach', 'doctor', 'medicine', 'treatment', 'symptom', 'sick', 'ill', 'hurt', 'ache'],
        'hi': ['दर्द', 'बुखार', 'सिरदर्द', 'खांसी', 'सर्दी', 'पेट', 'डॉक्टर', 'दवाई', 'इलाज', 'बीमार'],
        'ta': ['வலி', 'காய்ச்சல்', 'தலைவலி', 'இருமல்', 'சளி', 'வயிறு', 'மருத்துவர்', 'மருந்து'],
        'te': ['నొప్పి', 'జ్వరం', 'తలనొప్పి', 'దగ్గు', 'జలుబు', 'కడుపు', 'డాక్టర్', 'మందు'],
        'kn': ['ನೋವು', 'ಜ್ವರ', 'ತಲೆನೋವು', 'ಕೆಮ್ಮು', 'ಶೀತ', 'ಹೊಟ್ಟೆ', 'ವೈದ್ಯರು', 'ಔಷಧಿ'],
    },
    'appointment': {
        'en': ['appointment', 'book', 'schedule', 'slot', 'available', 'timing', 'visit', 'meet doctor', 'see doctor'],
        'hi': ['अपॉइंटमेंट', 'बुक', 'समय', 'मिलना', 'डॉक्टर से मिलना'],
        'ta': ['நேரம்', 'முன்பதிவு', 'சந்திப்பு'],
        'te': ['అపాయింట్మెంట్', 'బుక్', 'సమయం'],
        'kn': ['ಅಪಾಯಿಂಟ್ಮೆಂಟ್', 'ಬುಕ್', 'ಸಮಯ', 'ಭೇಟಿ'],
    },
    'greeting': {
        'en': ['hello', 'hi', 'hey', 'good morning', 'good evening', 'namaste'],
        'hi': ['नमस्ते', 'नमस्कार', 'हैलो'],
        'ta': ['வணக்கம்', 'ஹலோ'],
        'te': ['నమస్కారం', 'హలో'],
        'kn': ['ನಮಸ್ಕಾರ', 'ಹಲೋ'],
    },
    'admin': {
        'en': ['bill', 'payment', 'insurance', 'cost', 'price', 'charge', 'report', 'record', 'visiting hours', 'parking'],
        'hi': ['बिल', 'पेमेंट', 'बीमा', 'रिपोर्ट'],
        'ta': ['பில்', 'கட்டணம்', 'காப்பீடு'],
        'te': ['బిల్', 'చెల్లింపు', 'బీమా'],
        'kn': ['ಬಿಲ್', 'ಪಾವತಿ', 'ವಿಮೆ'],
    }
}

# Urgency keywords - HIGH priority
URGENCY_HIGH = {
    'en': ['severe', 'extreme', 'unbearable', 'emergency', 'urgent', 'immediately', "can't breathe", 'chest pain', 'unconscious', 'bleeding', 'accident', 'critical', 'dying', 'collapsed'],
    'hi': ['गंभीर', 'बहुत', 'असहनीय', 'इमरजेंसी', 'तुरंत', 'सांस नहीं', 'छाती में दर्द', 'बेहोश', 'खून'],
    'ta': ['கடுமையான', 'அவசரம்', 'உடனடி', 'மூச்சு', 'நெஞ்சு வலி', 'மயக்கம்', 'இரத்தம்'],
    'te': ['తీవ్రమైన', 'అత్యవసరం', 'వెంటనే', 'ఊపిరి', 'ఛాతీ నొప్పి', 'స్పృహ', 'రక్తం'],
    'kn': ['ತೀವ್ರ', 'ತುರ್ತು', 'ತಕ್ಷಣ', 'ಉಸಿರು', 'ಎದೆ ನೋವು', 'ಪ್ರಜ್ಞೆ', 'ರಕ್ತ'],
}

# Critical symptoms that always require escalation
CRITICAL_SYMPTOMS = {
    'en': ['chest pain', 'heart attack', "can't breathe", 'difficulty breathing', 'unconscious', 'seizure', 'stroke', 'severe bleeding', 'choking', 'suicide', 'poisoning', 'overdose'],
    'hi': ['छाती में दर्द', 'हार्ट अटैक', 'सांस नहीं', 'बेहोश', 'दौरा', 'खून बह रहा', 'जहर'],
    'ta': ['நெஞ்சு வலி', 'மாரடைப்பு', 'மூச்சு திணறல்', 'மயக்கம்', 'வலிப்பு'],
    'te': ['ఛాతీ నొప్పి', 'గుండెపోటు', 'ఊపిరి ఆడటం లేదు', 'స్పృహ లేదు'],
    'kn': ['ಎದೆ ನೋವು', 'ಹೃದಯಾಘಾತ', 'ಉಸಿರಾಟ ತೊಂದರೆ', 'ಪ್ರಜ್ಞೆ ತಪ್ಪು'],
}

# Stress indicators
STRESS_INDICATORS = {
    'en': ['help me', 'please help', 'very worried', 'scared', 'afraid', 'anxious', "can't sleep", 'desperate', 'terrible', 'worst pain', 'unbearable'],
    'hi': ['मदद करो', 'बहुत चिंता', 'डर', 'नींद नहीं', 'असहनीय'],
    'ta': ['உதவி', 'பயம்', 'கவலை', 'தூக்கமின்மை'],
    'te': ['సహాయం', 'భయం', 'ఆందోళన'],
    'kn': ['ಸಹಾಯ', 'ಭಯ', 'ಚಿಂತೆ', 'ನಿದ್ರೆ ಬರುತ್ತಿಲ್ಲ'],
}

print("✅ Configuration loaded!")

## 🤖 Step 4: Load ML Models

This cell loads all the AI models. It may take a few minutes on first run.

In [ ]:
# ============================================================================
# MODEL LOADING
# ============================================================================

class ModelState:
    """Global state for loaded models"""
    vad_model = None
    vad_utils = None
    whisper_model = None
    llm_model = None
    llm_tokenizer = None
    tts_model = None
    tts_tokenizer = None
    tts_description_tokenizer = None
    models_loaded = False

models = ModelState()

def load_vad_model():
    """Load Silero VAD model"""
    print("📢 Loading Silero VAD...")
    start = time.time()
    model, utils = torch.hub.load(
        repo_or_dir='snakers4/silero-vad',
        model='silero_vad',
        force_reload=False,
        onnx=False
    )
    model = model.to(DEVICE)
    print(f"   ✅ VAD loaded in {time.time() - start:.1f}s")
    return model, utils

def load_whisper_model():
    """Load Faster-Whisper STT model"""
    print("🎤 Loading Faster-Whisper (large-v3)...")
    start = time.time()
    from faster_whisper import WhisperModel
    
    if DEVICE == "cuda":
        model = WhisperModel("large-v3", device="cuda", compute_type="float16")
    else:
        model = WhisperModel("large-v3", device="cpu", compute_type="int8")
    
    print(f"   ✅ Whisper loaded in {time.time() - start:.1f}s")
    return model

def load_llm_model():
    """Load LLaMA model with 4-bit quantization"""
    print("🧠 Loading LLaMA 3.1-8B (4-bit quantized)...")
    start = time.time()
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    
    model_id = "meta-llama/Llama-3.1-8B-Instruct"
    
    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN
    )
    
    print(f"   ✅ LLaMA loaded in {time.time() - start:.1f}s")
    return model, tokenizer

def load_tts_model():
    """Load AI4Bharat Indic Parler-TTS model"""
    print("🔊 Loading Indic Parler-TTS...")
    start = time.time()
    from transformers import AutoTokenizer
    from parler_tts import ParlerTTSForConditionalGeneration
    
    model_id = "ai4bharat/indic-parler-tts"
    
    model = ParlerTTSForConditionalGeneration.from_pretrained(model_id).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    description_tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    
    print(f"   ✅ TTS loaded in {time.time() - start:.1f}s")
    return model, tokenizer, description_tokenizer

def load_all_models():
    """Load all models"""
    print("="*50)
    print("🚀 LOADING ALL MODELS")
    print("="*50)
    total_start = time.time()
    
    models.vad_model, models.vad_utils = load_vad_model()
    models.whisper_model = load_whisper_model()
    models.llm_model, models.llm_tokenizer = load_llm_model()
    models.tts_model, models.tts_tokenizer, models.tts_description_tokenizer = load_tts_model()
    models.models_loaded = True
    
    print("="*50)
    print(f"✅ ALL MODELS LOADED in {time.time() - total_start:.1f}s")
    print("="*50)

# Load all models
load_all_models()

## ⚙️ Step 5: Pipeline Functions (8 Stages)

In [ ]:
# ============================================================================
# STAGE 1: VOICE ACTIVITY DETECTION (VAD)
# ============================================================================

def run_vad(audio_data: np.ndarray, sample_rate: int) -> dict:
    """Run Voice Activity Detection"""
    start = time.time()
    
    # Resample to 16kHz if needed
    if sample_rate != 16000:
        import librosa
        audio_data = librosa.resample(audio_data, orig_sr=sample_rate, target_sr=16000)
        sample_rate = 16000
    
    # Convert to tensor
    audio_tensor = torch.FloatTensor(audio_data).to(DEVICE)
    
    # Run VAD
    get_speech_timestamps = models.vad_utils[0]
    speech_timestamps = get_speech_timestamps(audio_tensor, models.vad_model, sampling_rate=sample_rate)
    
    has_speech = len(speech_timestamps) > 0
    latency = (time.time() - start) * 1000
    
    return {
        "has_speech": has_speech,
        "speech_segments": speech_timestamps,
        "latency_ms": round(latency, 2)
    }

# ============================================================================
# STAGE 2: SPEECH-TO-TEXT (STT)
# ============================================================================

def run_stt(audio_data: np.ndarray, sample_rate: int, language: str = None) -> dict:
    """Run Speech-to-Text with Faster-Whisper"""
    start = time.time()
    
    # Save to temp file for Whisper
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        sf.write(f.name, audio_data, sample_rate)
        temp_path = f.name
    
    try:
        segments, info = models.whisper_model.transcribe(
            temp_path,
            language=language,
            beam_size=5,
            vad_filter=True
        )
        
        # Collect all segments
        text = " ".join([seg.text for seg in segments])
        detected_language = info.language
        confidence = info.language_probability
        
    finally:
        os.unlink(temp_path)
    
    latency = (time.time() - start) * 1000
    
    return {
        "text": text.strip(),
        "detected_language": detected_language,
        "confidence": round(confidence, 3),
        "latency_ms": round(latency, 2)
    }

# ============================================================================
# STAGE 3: LANGUAGE DETECTION (Script-Based)
# ============================================================================

def detect_script_language(text: str) -> dict:
    """Detect language from Unicode script"""
    start = time.time()
    
    char_counts = {lang: 0 for lang in SCRIPT_RANGES}
    
    for char in text:
        code_point = ord(char)
        for lang, ranges in SCRIPT_RANGES.items():
            for range_start, range_end in ranges:
                if range_start <= code_point <= range_end:
                    char_counts[lang] += 1
                    break
    
    total = sum(char_counts.values())
    if total == 0:
        detected = 'en'
        confidence = 0.5
    else:
        detected = max(char_counts, key=char_counts.get)
        confidence = char_counts[detected] / total
    
    latency = (time.time() - start) * 1000
    
    return {
        "language": detected,
        "language_name": SUPPORTED_LANGUAGES.get(detected, "Unknown"),
        "confidence": round(confidence, 3),
        "latency_ms": round(latency, 2)
    }

# ============================================================================
# STAGE 4: LAYER-1 SIGNAL EXTRACTION
# ============================================================================

def extract_layer1_signals(text: str, language: str) -> dict:
    """Extract intent, urgency, tone, and stress signals"""
    start = time.time()
    
    text_lower = text.lower()
    
    # Detect intent
    intent = "general"
    intent_confidence = 0.5
    
    for intent_type, lang_keywords in INTENT_KEYWORDS.items():
        keywords = lang_keywords.get(language, []) + lang_keywords.get('en', [])
        matches = sum(1 for kw in keywords if kw.lower() in text_lower)
        if matches > 0:
            confidence = min(matches / 3, 1.0)
            if confidence > intent_confidence:
                intent = intent_type
                intent_confidence = confidence
    
    # Detect urgency
    urgency = "normal"
    urgency_count = 0
    urgency_keywords = URGENCY_HIGH.get(language, []) + URGENCY_HIGH.get('en', [])
    for kw in urgency_keywords:
        if kw.lower() in text_lower:
            urgency_count += 1
    
    if urgency_count >= 2:
        urgency = "high"
    elif urgency_count == 1:
        urgency = "medium"
    
    # Detect stress indicators
    stress_level = "calm"
    stress_keywords = STRESS_INDICATORS.get(language, []) + STRESS_INDICATORS.get('en', [])
    if any(kw.lower() in text_lower for kw in stress_keywords):
        stress_level = "stressed"
    
    # Detect tone
    if urgency == "high" or stress_level == "stressed" or text.count('!') > 0:
        tone = "stressed"
    else:
        tone = "calm"
    
    latency = (time.time() - start) * 1000
    
    return {
        "intent": intent,
        "intent_confidence": round(intent_confidence, 3),
        "urgency": urgency,
        "urgency_count": urgency_count,
        "tone": tone,
        "stress_level": stress_level,
        "latency_ms": round(latency, 2)
    }

# ============================================================================
# STAGE 5: SAFETY GATE
# ============================================================================

def run_safety_gate(text: str, language: str, signals: dict) -> dict:
    """
    Safety Gate with 5 escalation rules:
    1. Critical symptoms → always escalate
    2. Chest/breathing + medical query → escalate
    3. Low confidence + medical query → escalate
    4. High urgency (>=2 keywords) → escalate
    5. Repeated similar query → escalate (simplified in notebook)
    """
    start = time.time()
    
    text_lower = text.lower()
    should_escalate = False
    escalation_reason = None
    escalation_rules_triggered = []
    
    # Rule 1: Check for critical symptoms
    critical_keywords = CRITICAL_SYMPTOMS.get(language, []) + CRITICAL_SYMPTOMS.get('en', [])
    for symptom in critical_keywords:
        if symptom.lower() in text_lower:
            should_escalate = True
            escalation_reason = f"Critical symptom detected: {symptom}"
            escalation_rules_triggered.append("rule_1_critical_symptom")
            break
    
    # Rule 2: Chest/breathing + medical query
    breathing_keywords = ['chest', 'breathe', 'breathing', 'heart', 'छाती', 'सांस', 'ಎದೆ', 'ಉಸಿರು']
    has_breathing_concern = any(kw in text_lower for kw in breathing_keywords)
    if has_breathing_concern and signals['intent'] == 'medical_query':
        should_escalate = True
        if not escalation_reason:
            escalation_reason = "Chest/breathing concern with medical query"
        escalation_rules_triggered.append("rule_2_breathing_medical")
    
    # Rule 3: Low confidence on medical query
    if signals['intent'] == 'medical_query' and signals['intent_confidence'] < 0.3:
        should_escalate = True
        if not escalation_reason:
            escalation_reason = "Low confidence medical query - needs clarification"
        escalation_rules_triggered.append("rule_3_low_confidence")
    
    # Rule 4: High urgency (2+ urgency keywords)
    if signals.get('urgency_count', 0) >= 2:
        should_escalate = True
        if not escalation_reason:
            escalation_reason = "Multiple urgency indicators detected"
        escalation_rules_triggered.append("rule_4_high_urgency")
    
    latency = (time.time() - start) * 1000
    
    return {
        "should_escalate": should_escalate,
        "escalation_reason": escalation_reason,
        "rules_triggered": escalation_rules_triggered,
        "latency_ms": round(latency, 2)
    }

# ============================================================================
# STAGE 6: POLICY ENGINE
# ============================================================================

def apply_policy(signals: dict, safety: dict, language: str) -> dict:
    """Apply hospital policy constraints"""
    start = time.time()
    
    config = HOSPITAL_CONFIG
    
    policy = {
        "hospital_name": config["hospital_name"],
        "response_language": language,
        "max_words": config["max_words"],
        "tone": config["tone"],
        "include_doctor_recommendation": config["always_recommend_doctor"],
        "emergency_number": config["emergency_number"],
        "helpline": config["helpline"],
        "disclaimer": config["disclaimer_required"],
        "disclaimer_text": config["disclaimer_text"]
    }
    
    # Adjust policy based on signals
    if safety['should_escalate']:
        policy['priority'] = 'urgent'
        policy['include_emergency_info'] = True
        policy['max_words'] = 80  # Allow longer response for emergencies
    else:
        policy['priority'] = 'normal'
        policy['include_emergency_info'] = False
    
    latency = (time.time() - start) * 1000
    policy['latency_ms'] = round(latency, 2)
    
    return policy

# ============================================================================
# STAGE 7: LLM RESPONSE GENERATION
# ============================================================================

def generate_llm_response(text: str, language: str, signals: dict, safety: dict, policy: dict) -> dict:
    """Generate response using LLaMA"""
    start = time.time()
    
    lang_name = SUPPORTED_LANGUAGES.get(language, "English")
    
    # Build system prompt with policy
    system_prompt = f"""You are a helpful medical assistant for {policy['hospital_name']}. 
Follow these rules strictly:
1. Respond in {lang_name} language using the native script
2. Keep responses under {policy['max_words']} words
3. Use a {policy['tone']} tone
4. Always recommend consulting a doctor for medical issues
5. For emergencies, mention calling {policy['emergency_number']}
6. Never diagnose or prescribe medication
7. Be empathetic and professional
8. Helpline: {policy['helpline']}

Patient intent: {signals['intent']}
Urgency level: {signals['urgency']}
{"URGENT: This may be an emergency situation. Prioritize safety and recommend immediate medical attention." if safety['should_escalate'] else ""}
"""
    
    # Add disclaimer instruction
    if policy.get('disclaimer'):
        system_prompt += f"\n\nInclude a brief disclaimer: {policy['disclaimer_text']}"

    # Build prompt
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text}
    ]
    
    # Format for LLaMA
    prompt = models.llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = models.llm_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        outputs = models.llm_model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=models.llm_tokenizer.eos_token_id
        )
    
    response = models.llm_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    latency = (time.time() - start) * 1000
    
    return {
        "response": response.strip(),
        "latency_ms": round(latency, 2)
    }

# ============================================================================
# STAGE 8: TEXT-TO-SPEECH (TTS)
# ============================================================================

def generate_tts(text: str, language: str) -> dict:
    """Generate speech using Indic Parler-TTS"""
    start = time.time()
    
    lang_name = SUPPORTED_LANGUAGES.get(language, "Hindi")
    
    # Description for the TTS voice
    description = f"A female speaker delivers a clear, professional medical response in {lang_name} with a calm and reassuring tone."
    
    # Tokenize
    description_tokens = models.tts_description_tokenizer(description, return_tensors="pt").to(DEVICE)
    prompt_tokens = models.tts_tokenizer(text, return_tensors="pt").to(DEVICE)
    
    # Generate audio
    with torch.no_grad():
        generation = models.tts_model.generate(
            input_ids=description_tokens.input_ids,
            attention_mask=description_tokens.attention_mask,
            prompt_input_ids=prompt_tokens.input_ids,
            prompt_attention_mask=prompt_tokens.attention_mask
        )
    
    audio_array = generation.cpu().numpy().squeeze()
    sample_rate = models.tts_model.config.sampling_rate
    
    latency = (time.time() - start) * 1000
    
    return {
        "audio": audio_array,
        "sample_rate": sample_rate,
        "duration_ms": round(len(audio_array) / sample_rate * 1000, 2),
        "latency_ms": round(latency, 2)
    }

print("✅ All pipeline functions defined!")

## 🎯 Step 6: Main Process Function

In [ ]:
def process_audio_file(audio_path: str, preferred_language: str = None) -> dict:
    """
    Process audio file through the complete 8-stage pipeline.
    
    Args:
        audio_path: Path to the audio file (WAV, MP3, etc.)
        preferred_language: Optional language code (kn, ta, te, hi, en)
    
    Returns:
        dict with transcription, response, audio, and pipeline details
    """
    if not models.models_loaded:
        return {"success": False, "error": "Models not loaded. Run the model loading cell first."}
    
    total_start = time.time()
    
    try:
        # Read audio file
        audio_data, sample_rate = sf.read(audio_path)
        
        # Convert stereo to mono if needed
        if len(audio_data.shape) > 1:
            audio_data = audio_data.mean(axis=1)
        
        # Ensure float32
        audio_data = audio_data.astype(np.float32)
        
        print("🎵 Audio loaded:", f"{len(audio_data)/sample_rate:.1f}s at {sample_rate}Hz")
        
        # ====== PIPELINE ======
        
        # 1. VAD
        print("\n📢 Stage 1: Voice Activity Detection...")
        vad_result = run_vad(audio_data, sample_rate)
        print(f"   Speech detected: {vad_result['has_speech']} ({vad_result['latency_ms']:.0f}ms)")
        
        if not vad_result['has_speech']:
            return {
                "success": False,
                "error": "No speech detected in audio",
                "pipeline": {"vad": vad_result}
            }
        
        # 2. STT
        print("\n🎤 Stage 2: Speech-to-Text...")
        stt_result = run_stt(audio_data, sample_rate, preferred_language)
        print(f"   Transcription: \"{stt_result['text']}\"")
        print(f"   Detected language: {stt_result['detected_language']} ({stt_result['latency_ms']:.0f}ms)")
        
        if not stt_result['text']:
            return {
                "success": False,
                "error": "Could not transcribe audio",
                "pipeline": {"vad": vad_result, "stt": stt_result}
            }
        
        # 3. Language Detection
        print("\n🌐 Stage 3: Language Detection...")
        lang_result = detect_script_language(stt_result['text'])
        detected_lang = preferred_language or lang_result['language']
        print(f"   Detected: {lang_result['language_name']} ({lang_result['latency_ms']:.0f}ms)")
        
        # 4. Layer-1 Signals
        print("\n📊 Stage 4: Signal Extraction...")
        signals_result = extract_layer1_signals(stt_result['text'], detected_lang)
        print(f"   Intent: {signals_result['intent']} | Urgency: {signals_result['urgency']} | Tone: {signals_result['tone']}")
        
        # 5. Safety Gate
        print("\n🚨 Stage 5: Safety Gate...")
        safety_result = run_safety_gate(stt_result['text'], detected_lang, signals_result)
        if safety_result['should_escalate']:
            print(f"   ⚠️  ESCALATION TRIGGERED: {safety_result['escalation_reason']}")
        else:
            print(f"   ✅ No escalation needed")
        
        # 6. Policy Engine
        print("\n📋 Stage 6: Policy Engine...")
        policy_result = apply_policy(signals_result, safety_result, detected_lang)
        print(f"   Priority: {policy_result['priority']} | Max words: {policy_result['max_words']}")
        
        # 7. LLM Response
        print("\n🧠 Stage 7: LLM Response Generation...")
        llm_result = generate_llm_response(
            stt_result['text'], 
            detected_lang, 
            signals_result, 
            safety_result, 
            policy_result
        )
        print(f"   Response: \"{llm_result['response'][:100]}...\"" if len(llm_result['response']) > 100 else f"   Response: \"{llm_result['response']}\"")
        print(f"   ({llm_result['latency_ms']:.0f}ms)")
        
        # 8. TTS
        print("\n🔊 Stage 8: Text-to-Speech...")
        tts_result = generate_tts(llm_result['response'], detected_lang)
        print(f"   Audio generated: {tts_result['duration_ms']:.0f}ms ({tts_result['latency_ms']:.0f}ms)")
        
        # Calculate total latency
        total_latency = (time.time() - total_start) * 1000
        
        print("\n" + "="*50)
        print(f"✅ PIPELINE COMPLETE in {total_latency:.0f}ms")
        print("="*50)
        
        return {
            "success": True,
            "transcription": stt_result['text'],
            "response": llm_result['response'],
            "audio": tts_result['audio'],
            "audio_sample_rate": tts_result['sample_rate'],
            "language": detected_lang,
            "language_name": SUPPORTED_LANGUAGES.get(detected_lang, "Unknown"),
            "escalation": {
                "should_escalate": safety_result['should_escalate'],
                "reason": safety_result['escalation_reason']
            },
            "pipeline": {
                "vad_ms": vad_result['latency_ms'],
                "stt_ms": stt_result['latency_ms'],
                "llm_ms": llm_result['latency_ms'],
                "tts_ms": tts_result['latency_ms']
            },
            "total_latency_ms": round(total_latency, 2)
        }
        
    except Exception as e:
        import traceback
        print(f"\n❌ Error: {e}")
        traceback.print_exc()
        return {"success": False, "error": str(e)}

print("✅ Main process function ready!")

## 🎤 Step 7: Simple Audio Upload Interface

Upload an audio file and get the AI response!

In [ ]:
from ipywidgets import FileUpload, Button, Output, VBox, HBox, Dropdown, HTML
from IPython.display import display, Audio, clear_output
import tempfile

# Create widgets
upload_widget = FileUpload(
    accept='.wav,.mp3,.m4a,.ogg,.flac',
    multiple=False,
    description='Upload Audio'
)

language_dropdown = Dropdown(
    options=[('Auto-detect', None)] + [(v, k) for k, v in SUPPORTED_LANGUAGES.items()],
    value=None,
    description='Language:'
)

process_button = Button(
    description='🎯 Process Audio',
    button_style='success',
    layout={'width': '150px'}
)

output_area = Output()

def on_process_click(b):
    with output_area:
        clear_output(wait=True)
        
        if not upload_widget.value:
            print("⚠️ Please upload an audio file first!")
            return
        
        # Get uploaded file
        uploaded_file = list(upload_widget.value.values())[0]
        file_content = uploaded_file['content']
        file_name = list(upload_widget.value.keys())[0]
        
        print(f"📁 Processing: {file_name}")
        print("="*50)
        
        # Save to temp file
        suffix = '.' + file_name.split('.')[-1] if '.' in file_name else '.wav'
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
            f.write(file_content)
            temp_path = f.name
        
        try:
            # Process audio
            result = process_audio_file(temp_path, language_dropdown.value)
            
            if result['success']:
                print("\n" + "="*50)
                print("📝 TRANSCRIPTION:")
                print("="*50)
                print(result['transcription'])
                
                print("\n" + "="*50)
                print("💬 AI RESPONSE:")
                print("="*50)
                print(result['response'])
                
                if result['escalation']['should_escalate']:
                    print("\n⚠️  ESCALATION:", result['escalation']['reason'])
                
                print("\n" + "="*50)
                print("🔊 AUDIO RESPONSE:")
                print("="*50)
                display(Audio(result['audio'], rate=result['audio_sample_rate'], autoplay=True))
                
            else:
                print(f"\n❌ Error: {result['error']}")
                
        finally:
            os.unlink(temp_path)

process_button.on_click(on_process_click)

# Display interface
display(HTML("<h3>🎤 Upload Audio File</h3>"))
display(HBox([upload_widget, language_dropdown, process_button]))
display(output_area)

## 📁 Alternative: Process Audio File Directly

If you prefer to process a file by path instead of uploading:

In [ ]:
# Example: Process an audio file directly by path
# Uncomment and modify the path to test:

# audio_path = "/path/to/your/audio.wav"
# result = process_audio_file(audio_path)

# if result['success']:
#     print("\nTranscription:", result['transcription'])
#     print("\nResponse:", result['response'])
#     display(Audio(result['audio'], rate=result['audio_sample_rate'], autoplay=True))

## 📊 Pipeline Statistics

Run this cell after processing to see performance breakdown:

In [ ]:
# Check if last result is available (from the process function above)
if 'result' in dir() and result.get('success'):
    print("📊 PIPELINE LATENCY BREAKDOWN")
    print("="*40)
    p = result['pipeline']
    print(f"VAD:  {p['vad_ms']:>8.0f} ms")
    print(f"STT:  {p['stt_ms']:>8.0f} ms")
    print(f"LLM:  {p['llm_ms']:>8.0f} ms")
    print(f"TTS:  {p['tts_ms']:>8.0f} ms")
    print("-"*40)
    print(f"TOTAL:{result['total_latency_ms']:>8.0f} ms")
else:
    print("⚠️ No result available. Process an audio file first!")